# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Tự tìm repo dù kernel đang ở trong repo, /content hay /kaggle/working.
SUBMISSION_REL = Path("submissions/2A202602451_TruongThiLanAnh")
starts = [Path.cwd(), *Path.cwd().parents, Path("/content"), Path("/kaggle/working")]
candidates = []
for start in starts:
    if start.exists():
        candidates.append(start)
        candidates.extend(p.parent for p in start.glob("*/eval.py"))
REPO_ROOT = next((p.resolve() for p in candidates
                  if (p / "eval.py").is_file() and (p / SUBMISSION_REL / "code").is_dir()), None)
if REPO_ROOT is None:
    # Colab mở notebook trực tiếp không tự clone toàn bộ repo.
    base = Path("/content") if Path("/content").exists() else Path.cwd()
    target = base / "K4-Track4-Day2-Deeplearning-Advance"
    if (target / ".git").is_dir():
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only", "origin", "main"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/SxAinsworth/K4-Track4-Day2-Deeplearning-Advance.git", str(target)], check=True)
    REPO_ROOT = target.resolve()
REPO_DIR = str(REPO_ROOT)
SUBMISSION_DIR = str(REPO_ROOT / SUBMISSION_REL)
os.chdir(REPO_ROOT)  # mọi đường dẫn data/, runs/, predictions/ tính từ gốc repo
sys.path.insert(0, str(REPO_ROOT / SUBMISSION_REL / "code"))
sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)
print("CODE_DIR:", REPO_ROOT / SUBMISSION_REL / "code")
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

# Lưu checkpoint/log bền vững trên Google Drive khi chạy Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/DeepWeeds_Day2_2A202602451")
except ImportError:
    ARTIFACT_ROOT = Path(SUBMISSION_DIR)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib, subprocess

ZIP = "data/images.zip"
URL = "https://zenodo.org/records/7939060/files/images.zip?download=1"
EXPECTED_MD5, EXPECTED_SIZE = "b7b30f96d466fba86016aa5a26606e0f", 491_516_047  # theo Zenodo API
os.makedirs("data/labels", exist_ok=True)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# File hỏng từ lần trước (tải dở hoặc trang lỗi HTML) sẽ bị xoá; wget -c tải tiếp phần còn thiếu.
for attempt in range(1, 6):
    size = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
    if size == EXPECTED_SIZE and md5(ZIP) == EXPECTED_MD5:
        print("images.zip OK:", size, "bytes, MD5", EXPECTED_MD5)
        break
    with open(ZIP, "rb") if size else open(os.devnull, "rb") as f:
        is_zip = f.read(2) == b"PK"
    if size >= EXPECTED_SIZE or (size and not is_zip):   # MD5 sai hoặc không phải zip (trang lỗi HTML) -> tải lại từ đầu
        print(f"Lần {attempt}: file hỏng ({size} bytes, zip={is_zip}), xoá và tải lại"); os.remove(ZIP)
    elif size:
        print(f"Lần {attempt}: mới có {size}/{EXPECTED_SIZE} bytes, tải tiếp")
    subprocess.run(["wget", "-c", "--tries=3", "--timeout=60", "-O", ZIP, URL], check=False)
else:
    raise RuntimeError(f"Không tải được images.zip đúng MD5 sau 5 lần (size={os.path.getsize(ZIP)})")

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels


In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
from step0_checks import run_eda
from IPython.display import display, Image as DisplayImage

EDA_DIR = f"{SUBMISSION_DIR}/eda"
eda_report = run_eda(LABELS_DIR, IMAGES_DIR, EDA_DIR)
print("Tỷ lệ lớp lớn nhất/nhỏ nhất:", eda_report["imbalance_ratio"])
display(DisplayImage(filename=f"{EDA_DIR}/class_distribution.png"))
display(DisplayImage(filename=f"{EDA_DIR}/samples_3_per_class.png"))
display(DisplayImage(filename=f"{EDA_DIR}/augmented_samples.png"))

In [ ]:
from step0_checks import math_checks, initial_loss_check, mode_checks, overfit_small_batch
from train import Config, run, set_seed

set_seed(0)
print("Kiểm tra CE / focal / CutMix:", math_checks())
# Loss ban đầu của model thật (backbone pretrained + head 9 lớp mới) trên ảnh val, kỳ vọng ≈ ln 9 = 2.197
for b in ["resnet50", "convnext_tiny", "efficientnet_b0", "mobilenetv3_large_100", "deit_tiny_patch16_224"]:
    print("Loss ban đầu:", initial_loss_check(b, LABELS_DIR, IMAGES_DIR))
print("train()/eval() và BN đóng băng:", mode_checks())
print("Overfit batch nhỏ:", overfit_small_batch("resnet18", LABELS_DIR, IMAGES_DIR))
# Ảnh augmentation đã được tạo và hiển thị ở ô EDA phía trên.
# Smoke run nhanh (không dùng số này làm kết quả thí nghiệm):
smoke = run(Config(exp_id="SMOKE", backbone="resnet18", init="scratch", epochs=1,
                   batch_size=16, img_size=128, num_workers=0, amp=False,
                   debug_train_samples=64, debug_val_samples=64,
                   out_dir="runs", pred_dir="runs/smoke_predictions", curves_dir=EDA_DIR))
print(smoke)


In [ ]:
# Unit test tự viết (focal γ=0, CutMix, param_groups, BN đóng băng, thứ tự evaluate) và bộ test của repo
!cd {SUBMISSION_DIR}/code && PYTHONUTF8=1 python -m unittest test_step0 test_step1 test_step2 test_step3 test_step4 -v
!cd {REPO_DIR} && PYTHONUTF8=1 python -m unittest discover -s tests


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from step1_backbones import BACKBONES, run_screening

# T00 giống hệt cho cả 5 model: 12 epoch, seed 0, effective batch 64, AMP.
# Nếu Colab ngắt, chạy lại ô này: model đã hoàn tất sẽ tự được bỏ qua.
assert torch.cuda.is_available(), "Bước 1 cần GPU; hãy chọn Runtime > Change runtime type > T4 GPU"
step1 = run_screening(ARTIFACT_ROOT, epochs=12, batch_size=64, grad_accum_steps=1, seed=0)
display(step1["table"])
print("Chọn backbone:", step1["selection"])
print("Workbook:", step1["results_xlsx"])

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
from step2_training import run_training_ablations

# T00 chạy seed 0/1/2 để đo nhiễu; ablation dùng seed 0 và chỉ đổi một yếu tố.
# Bao gồm A (khởi tạo), B (augmentation), C (loss), F (EMA) và T11 kết hợp.
step2 = run_training_ablations(ARTIFACT_ROOT, epochs=12, batch_size=64)
display(step2["table"])
print("Công thức chọn cho Bước 3:", step2["selection"])
print("Workbook:", step2["results_xlsx"])

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
from step3_inference import run_inference_comparison

step3 = run_inference_comparison(ARTIFACT_ROOT, LABELS_DIR, IMAGES_DIR)
display(step3["inference"])
display(step3["latency"])
print("Lựa chọn Bước 3:", step3["selection"])
print("Workbook:", step3["results_xlsx"])

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
from step4_final import run_final

# CẢNH BÁO: ô này là lần duy nhất mở test. Không đổi cấu hình sau khi chạy.
step4 = run_final(ARTIFACT_ROOT, LABELS_DIR, IMAGES_DIR, epochs=12, batch_size=64)
display(step4["final"])
display(step4["hard_classes"])
display(step4["temperatures"])
print("Khóa cấu hình:", step4["lock"])
print("Kết quả eval.py:", step4["eval_log"])

In [ ]:
# run_final đã gọi eval.py score cho F01 và T00 bằng các file trong Google Drive.
print(Path(step4["eval_log"]).read_text(encoding="utf-8"))

In [ ]:
# Điểm đề xuất I1–I5 đã gồm uncalibrated, final-val và latency p95 đo đúng cách.
import json
display(json.loads((Path(ARTIFACT_ROOT) / "eval_out/grade_I.json").read_text(encoding="utf-8")))

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py